In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D3 — Occupational Employment and Wages — May 2024
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip install pymupdf -q

from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import fitz

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D3"

DOCUMENT_NAME = (
    "Occupational Employment and Wages — May 2024"
)

BRANCH = "A"
BRANCH_NAME = "Direct Ingestion"

INPUT_REPRESENTATION = "Original PDF"

EXPECTED_SOURCE_FORMAT = ".pdf"

REFERENCE_PERIOD = "May 2024"

SOURCE_PAGE_START = 1
SOURCE_PAGE_END = 5

EXPECTED_PAGE_COUNT = 23
EXPECTED_RECORD_COUNT = 70

EXPECTED_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Value",
    "Unit",
    "Reference Period"
]

ALLOWED_UNITS = [
    "workers",
    "million workers",
    "percent",
    "USD"
]

EXPECTED_CONTENT_MARKERS = [
    "OCCUPATIONAL EMPLOYMENT AND WAGES",
    "Production occupations",
    "Architecture and engineering occupations",
    (
        "Building and grounds cleaning "
        "and maintenance occupations"
    ),
    "Largest occupations",
    "Public sector occupations",
    "May 2024"
]

OUTPUT_DIR = Path(
    "outputs_D3_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Document:",
    DOCUMENT_ID
)

print(
    "Branch:",
    BRANCH
)

print(
    "Output directory:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# 2. Source document upload
# ============================================================

print(
    "Upload the original D3 PDF:\n"
    "D3 - ocwage.pdf"
)

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:

    raise ValueError(
        "Upload exactly one PDF file."
    )

PDF_PATH = pdf_files[0]

print(
    "Uploaded PDF:",
    PDF_PATH.name
)

In [ ]:
# ============================================================
# 3. Source SHA-256
# ============================================================

def sha256_file(path):
    """
    Return the SHA-256 hash of a file.
    """

    hash_object = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            hash_object.update(
                chunk
            )

    return hash_object.hexdigest()


PDF_SHA256 = sha256_file(
    PDF_PATH
)

print(
    "PDF SHA-256:",
    PDF_SHA256
)

In [ ]:
# ============================================================
# 4. Source PDF diagnostics
# ============================================================

pdf_document = fitz.open(
    PDF_PATH
)

PAGE_COUNT = len(
    pdf_document
)

page_texts = []

for page_index, page in enumerate(
    pdf_document
):

    page_texts.append({
        "page_number":
            page_index + 1,

        "text":
            page.get_text(
                "text"
            )
    })


FULL_PDF_TEXT = "\n".join(
    page_record["text"]
    for page_record
    in page_texts
)


if not FULL_PDF_TEXT.strip():

    raise ValueError(
        "No directly extractable text was found."
    )


print(
    "Page count:",
    PAGE_COUNT
)

print(
    "Text characters:",
    len(FULL_PDF_TEXT)
)

print(
    "Text words:",
    len(
        FULL_PDF_TEXT.split()
    )
)

In [ ]:
# ============================================================
# 5. Extraction-scope diagnostics
# ============================================================

EXTRACTION_SCOPE_TEXT = "\n".join(
    page_record["text"]

    for page_record
    in page_texts

    if (
        SOURCE_PAGE_START
        <= page_record[
            "page_number"
        ]
        <= SOURCE_PAGE_END
    )
)


print(
    "Extraction-scope characters:",
    len(
        EXTRACTION_SCOPE_TEXT
    )
)

print(
    "\nPages 1–5 preview:\n"
)

print(
    EXTRACTION_SCOPE_TEXT[:3000]
)

In [ ]:
# ============================================================
# 6. Source integrity checks
# ============================================================

pdf_non_empty = (
    PDF_PATH.exists()
    and PDF_PATH.stat().st_size > 0
)

page_count_valid = (
    PAGE_COUNT == EXPECTED_PAGE_COUNT
)

text_extractable = bool(
    FULL_PDF_TEXT.strip()
)

extraction_scope_non_empty = bool(
    EXTRACTION_SCOPE_TEXT.strip()
)


expected_content_markers = {
    marker:
        marker.casefold()
        in EXTRACTION_SCOPE_TEXT.casefold()

    for marker
    in EXPECTED_CONTENT_MARKERS
}


all_expected_content_markers_present = all(
    expected_content_markers.values()
)


original_pdf_usable = all([
    pdf_non_empty,
    page_count_valid,
    text_extractable,
    extraction_scope_non_empty,
    all_expected_content_markers_present
])


print(
    "PDF non-empty:",
    pdf_non_empty
)

print(
    "Page count valid:",
    page_count_valid
)

print(
    "Text extractable:",
    text_extractable
)

print(
    "Pages 1–5 non-empty:",
     extraction_scope_non_empty
)

print(
    "Expected markers:"
)

print(
    json.dumps(
        expected_content_markers,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "Original PDF usable:",
    original_pdf_usable
)


if not original_pdf_usable:

    raise ValueError(
        "The original PDF failed the "
        "Branch A integrity checks."
    )

In [ ]:
# ============================================================
# 7. Source integrity report
# ============================================================

ORIGINAL_PDF_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "input_file":
        PDF_PATH.name,

    "input_representation":
        INPUT_REPRESENTATION,

    "file_sha256":
        PDF_SHA256,

    "file_size_bytes":
        PDF_PATH.stat().st_size,

    "observed_page_count":
        PAGE_COUNT,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "page_count_valid":
        page_count_valid,

    "text_extractable":
        text_extractable,

    "ocr_required":
        False,

    "extraction_scope_pages": [
        SOURCE_PAGE_START,
        SOURCE_PAGE_END
    ],

    "extraction_scope_character_count":
        len(
            EXTRACTION_SCOPE_TEXT
        ),

    "expected_content_markers":
        expected_content_markers,

    "all_expected_content_markers_present":
        all_expected_content_markers_present,

    "direct_pdf_ingestion_usable":
        original_pdf_usable,

    "input_integrity_passed":
        original_pdf_usable
}


ORIGINAL_PDF_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_input_integrity.json"
)


with open(
    ORIGINAL_PDF_INTEGRITY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        ORIGINAL_PDF_INTEGRITY,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        ORIGINAL_PDF_INTEGRITY,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 8. Branch A representation
# ============================================================

BRANCH_REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        PDF_PATH.name,

    "input_format":
        PDF_PATH.suffix.lower(),

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "diagnostic_text_extraction_applied":
        True,

    "derived_representation_used_as_model_input":
        False,

    "model_input_description": (
        "The original 23-page PDF document is "
        "submitted directly to the LLM. "
        "The extraction task restricts the "
        "target scope to pages 1–5, but the PDF "
        "itself is not cropped or transformed."
    )
}


REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_representation.json"
)


with open(
    REPRESENTATION_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        BRANCH_REPRESENTATION,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        BRANCH_REPRESENTATION,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 9. Extraction schema
# ============================================================

EXPECTED_OUTPUT_STRUCTURE = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "records": [
        {
            "Section": None,
            "Indicator": None,
            "Occupation or Group": None,
            "Value": None,
            "Unit": None,
            "Reference Period": None
        }
    ]
}


print(
    json.dumps(
        EXPECTED_OUTPUT_STRUCTURE,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 10. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every explicitly stated occupational statistic from the headline
narrative sections on pages 1–5 of the attached original PDF document.

Return one record for every statistic represented within the defined
scope.

For each record, extract:

- Section
- Indicator
- Occupation or Group
- Value
- Unit
- Reference Period

Scope and extraction rules:

- Treat the attached original PDF as the only source of information.
- Use only the headline narrative sections on pages 1–5.
- Extract only information explicitly supported by the document.
- Do not extract the release identifier, release date, contact details,
  website addresses or other publication metadata.
- Do not extract general programme-description counts.
- Do not extract the Technical Note.
- Do not extract the full multi-page Table 1.
- Do not create separate observations from charts when the same values
  are already stated in the narrative.
- Do not include headings without numerical observations as records.
- Do not calculate, infer, reconstruct, aggregate, correct or invent
  any value.
- Do not increase the precision of rounded values.
- Preserve rounded employment values in the reported scale. For example,
  "8.7 million" must be returned as Value 8.7 with Unit
  "million workers".
- Return exact employment counts as numerical values with Unit
  "workers".
- Return employment shares and concentration values as numerical
  values with Unit "percent".
- Return annual mean wages as numerical values with Unit "USD".
- Preserve the occupation, group, industry or location wording used in
  the headline narrative.
- Use "May 2024" as the Reference Period for every record.
- Return Value as a numerical value, not as formatted text.
- Use null only when a requested value is not available.
- Verify that only the defined headline narrative scope has been
  processed.
- Verify that every explicitly stated occupational statistic within
  that scope has been processed.
- Verify that rounded values remain in their reported scale.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
"""

In [ ]:
# ============================================================
# 11. Extraction prompt
# ============================================================

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The original 23-page PDF document is attached as the extraction source.

The extraction scope is restricted to the headline narrative sections
on pages 1–5. The PDF itself has not been cropped or transformed.

Return only the JSON object.
""".strip()


PROMPT_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_prompt.txt"
)


PROMPT_PATH.write_text(
    FULL_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = hashlib.sha256(
    FULL_PROMPT.encode(
        "utf-8"
    )
).hexdigest()


print(FULL_PROMPT)

print(
    "\nPrompt saved:",
    PROMPT_PATH
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

In [ ]:
# ============================================================
# 12. Experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        PDF_PATH.name,

    "source_format":
        PDF_PATH.suffix.lower(),

    "source_sha256":
        PDF_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,

        "observed_page_count":
            PAGE_COUNT,

        "page_count_verified":
            page_count_valid,

        "machine_readable_text_layer":
            text_extractable,

        "fixed_extraction_scope_pages": [
            SOURCE_PAGE_START,
            SOURCE_PAGE_END
        ],

        "scope_content_markers_verified":
            all_expected_content_markers_present
    },

    "input_representation":
        "Original PDF document",

    "direct_document_ingestion":
        True,

    "structural_conversion_applied":
        False,

    "text_extraction_used_as_model_input":
        False,

    "diagnostic_text_extraction_applied":
        True,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "page_extraction_applied":
        False,

    "normalisation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "value_modification_applied":
        False,

    "value_rounding_applied":
        False,

    "derived_calculation_applied":
        False,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_fields":
            EXPECTED_FIELDS,

        "reference_period":
            REFERENCE_PERIOD
    },

    "allowed_units":
        ALLOWED_UNITS,

    "excluded_document_regions": [
        "Release metadata",
        "General OEWS programme-description counts",
        "Technical Note",
        "Full Table 1",
        "Repeated chart observations"
    ],

    "input_integrity_file":
        ORIGINAL_PDF_INTEGRITY_PATH.name,

    "input_integrity_passed":
        bool(
            ORIGINAL_PDF_INTEGRITY[
                "input_integrity_passed"
            ]
        ),

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        "JSON",

    "execution_environment":
        "Independent ChatGPT conversation",

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "notes": (
        "Branch A submits the original 23-page PDF "
        "directly to the model. Diagnostic text extraction "
        "is used only to verify source integrity. The fixed "
        "Stage 1 extraction task restricts the target scope "
        "to headline narrative statistics on pages 1–5. "
        "The document supplied to the model is not cropped "
        "or otherwise transformed."
    )
}


EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_experiment_metadata.json"
)


with open(
    EXPERIMENT_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_METADATA,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        EXPERIMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

## Independent Branch A extraction

Open new independent ChatGPT conversation.

Upload:

1. the original `D3 - ocwage.pdf`;
2. `D3_branch_A_prompt.txt`.

Submit the exact saved prompt once.

Save complete model response exactly as returned.


In [ ]:
# ============================================================
# 13. Raw response upload
# ============================================================

uploaded_output = files.upload()


if len(
    uploaded_output
) != 1:

    raise ValueError(
        "Upload exactly one file containing "
        "the complete raw D3 Branch A LLM response."
    )


RAW_OUTPUT_SOURCE_PATH = Path(
    next(
        iter(
            uploaded_output
        )
    )
)


print(
    "Uploaded raw response:",
    RAW_OUTPUT_SOURCE_PATH.name
)

In [ ]:
# ============================================================
# 14. Raw-response preservation
# ============================================================

RAW_RESPONSE_TEXT = (
    RAW_OUTPUT_SOURCE_PATH
    .read_text(
        encoding="utf-8"
    )
)


if not RAW_RESPONSE_TEXT.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )


RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_raw_response.txt"
)


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = hashlib.sha256(
    RAW_RESPONSE_TEXT.encode(
        "utf-8"
    )
).hexdigest()


print(
    "Raw response preserved:",
    RAW_RESPONSE_PATH
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

In [ ]:
# ============================================================
# 15. Raw-response parsing
# ============================================================

valid_json = True
json_parsing_error = None
PARSED_EXTRACTION = None


try:

    PARSED_EXTRACTION = json.loads(
        RAW_RESPONSE_TEXT
    )


except json.JSONDecodeError as error:

    valid_json = False

    json_parsing_error = str(
        error
    )


print(
    "Valid JSON:",
    valid_json
)

print(
    "Parsing error:",
    json_parsing_error
)

In [ ]:
# ============================================================
# 16. Parsed extraction
# ============================================================

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_parsed_extraction.json"
)


if valid_json:

    with open(
        PARSED_EXTRACTION_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            PARSED_EXTRACTION,
            file,
            indent=2,
            ensure_ascii=False
        )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH
    )


else:

    print(
        "Parsed extraction was not created because "
        "the raw response is not valid JSON."
    )

In [ ]:
# ============================================================
# 17. Top-level structure diagnostics
# ============================================================

top_level_object_valid = False

document_id_present = False
document_id_correct = False

branch_present = False
branch_correct = False

records_present = False
records_is_list = False

extracted_records = []


if (
    valid_json
    and isinstance(
        PARSED_EXTRACTION,
        dict
    )
):

    top_level_object_valid = True

    document_id_present = (
        "document_id"
        in PARSED_EXTRACTION
    )

    document_id_correct = (
        PARSED_EXTRACTION.get(
            "document_id"
        )
        == DOCUMENT_ID
    )

    branch_present = (
        "branch"
        in PARSED_EXTRACTION
    )

    branch_correct = (
        PARSED_EXTRACTION.get(
            "branch"
        )
        == BRANCH
    )

    records_present = (
        "records"
        in PARSED_EXTRACTION
    )

    records_is_list = isinstance(
        PARSED_EXTRACTION.get(
            "records"
        ),
        list
    )

    if records_is_list:

        extracted_records = (
            PARSED_EXTRACTION[
                "records"
            ]
        )


number_of_records = len(
    extracted_records
)


record_count_valid = (
    number_of_records
    == EXPECTED_RECORD_COUNT
)


print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID present:",
    document_id_present
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch present:",
    branch_present
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records present:",
    records_present
)

print(
    "Records is list:",
    records_is_list
)

print(
    "Observed record count:",
    number_of_records
)

print(
    "Record count valid:",
    record_count_valid
)

In [ ]:
# ============================================================
# 18. Record-structure diagnostics
# ============================================================

record_structure_issues = []

for record_index, record in enumerate(
    extracted_records
):

    issues = []

    if not isinstance(
        record,
        dict
    ):

        issues.append(
            "Record is not a JSON object."
        )

    else:

        actual_fields = set(
            record.keys()
        )

        expected_fields = set(
            EXPECTED_FIELDS
        )

        missing_fields = sorted(
            expected_fields
            - actual_fields
        )

        extra_fields = sorted(
            actual_fields
            - expected_fields
        )

        if missing_fields:

            issues.append({
                "missing_fields":
                    missing_fields
            })

        if extra_fields:

            issues.append({
                "extra_fields":
                    extra_fields
            })

    if issues:

        record_structure_issues.append({
            "record_index":
                record_index,

            "issues":
                issues
        })


records_with_structure_issues = len(
    record_structure_issues
)


print(
    "Records with structure issues:",
    records_with_structure_issues
)

if record_structure_issues:

    print(
        json.dumps(
            record_structure_issues,
            indent=2,
            ensure_ascii=False
        )
    )

In [ ]:
# ============================================================
# 19. Field-type diagnostics
# ============================================================

field_type_issues = []

TEXT_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Unit",
    "Reference Period"
]


for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    for field in TEXT_FIELDS:

        value = record.get(
            field
        )

        if (
            value is not None
            and not isinstance(
                value,
                str
            )
        ):

            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    field,

                "observed_type":
                    type(
                        value
                    ).__name__
            })


    value = record.get(
        "Value"
    )

    if (
        value is not None
        and (
            not isinstance(
                value,
                (int, float)
            )
            or isinstance(
                value,
                bool
            )
        )
    ):

        field_type_issues.append({
            "record_index":
                record_index,

            "field":
                "Value",

            "observed_type":
                type(
                    value
                ).__name__
        })


records_with_type_issues = len({
    issue[
        "record_index"
    ]

    for issue
    in field_type_issues
})


print(
    "Records with type issues:",
    records_with_type_issues
)

if field_type_issues:

    print(
        json.dumps(
            field_type_issues,
            indent=2,
            ensure_ascii=False
        )
    )

In [ ]:
# ============================================================
# 20. Unit and reference-period diagnostics
# ============================================================

unit_issues = []
reference_period_issues = []

for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    observed_unit = record.get(
        "Unit"
    )

    observed_period = record.get(
        "Reference Period"
    )


    if (
        observed_unit is not None
        and observed_unit
        not in ALLOWED_UNITS
    ):

        unit_issues.append({
            "record_index":
                record_index,

            "occupation_or_group":
                record.get(
                    "Occupation or Group"
                ),

            "observed_unit":
                observed_unit
        })


    if (
        observed_period is not None
        and observed_period
        != REFERENCE_PERIOD
    ):

        reference_period_issues.append({
            "record_index":
                record_index,

            "occupation_or_group":
                record.get(
                    "Occupation or Group"
                ),

            "observed_reference_period":
                observed_period
        })


print(
    "Unexpected units:",
    len(
        unit_issues
    )
)

print(
    "Unexpected reference periods:",
    len(
        reference_period_issues
    )
)

In [ ]:
# ============================================================
# 21. Missing-value diagnostics
# ============================================================

missing_values_by_field = {
    field: sum(
        1

        for record
        in extracted_records

        if (
            not isinstance(
                record,
                dict
            )
            or record.get(
                field
            ) is None
        )
    )

    for field
    in EXPECTED_FIELDS
}


print(
    "Missing values by field:"
)

print(
    json.dumps(
        missing_values_by_field,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 22. Duplicate-record diagnostics
# ============================================================

def create_record_key(record):
    """
    Create a strict record key without modifying the
    extracted representation.
    """

    if not isinstance(
        record,
        dict
    ):

        return None

    return (
        record.get(
            "Section"
        ),
        record.get(
            "Indicator"
        ),
        record.get(
            "Occupation or Group"
        ),
        record.get(
            "Reference Period"
        )
    )


record_keys = [
    create_record_key(
        record
    )

    for record
    in extracted_records
]


duplicate_record_keys = []

for key in set(
    record_keys
):

    if (
        key is not None
        and record_keys.count(
            key
        ) > 1
    ):

        duplicate_record_keys.append(
            key
        )


duplicate_record_keys = sorted(
    duplicate_record_keys,
    key=lambda item: str(
        item
    )
)


print(
    "Duplicate record-key count:",
    len(
        duplicate_record_keys
    )
)

if duplicate_record_keys:

    print(
        json.dumps(
            duplicate_record_keys,
            indent=2,
            ensure_ascii=False
        )
    )

In [ ]:
# ============================================================
# 23. Excluded-content diagnostics
# ============================================================

EXCLUDED_INDICATOR_TERMS = [
    "release reference",
    "release date",
    "contact information"
]

EXCLUDED_SECTION_TERMS = [
    "technical note",
    "table 1"
]


excluded_content_issues = []

for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    indicator = str(
        record.get(
            "Indicator",
            ""
        )
    ).casefold()

    section = str(
        record.get(
            "Section",
            ""
        )
    ).casefold()


    matched_indicator_terms = [
        term

        for term
        in EXCLUDED_INDICATOR_TERMS

        if term in indicator
    ]


    matched_section_terms = [
        term

        for term
        in EXCLUDED_SECTION_TERMS

        if term in section
    ]


    if (
        matched_indicator_terms
        or matched_section_terms
    ):

        excluded_content_issues.append({
            "record_index":
                record_index,

            "section":
                record.get(
                    "Section"
                ),

            "indicator":
                record.get(
                    "Indicator"
                ),

            "matched_excluded_terms":
                (
                    matched_indicator_terms
                    + matched_section_terms
                )
        })


print(
    "Clearly excluded-content records:",
    len(
        excluded_content_issues
    )
)

In [ ]:
# ============================================================
# 24. Technical diagnostic summary
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    records_with_structure_issues == 0,
    records_with_type_issues == 0
])


TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "number_of_records":
        number_of_records,

    "record_count_valid":
        record_count_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        record_structure_issues,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issues":
        field_type_issues,

    "unexpected_unit_count": len(unit_issues),

    "unexpected_reference_period_count": len(reference_period_issues),

    "unit_issues":
        unit_issues,

    "reference_period_issues":
        reference_period_issues,

    "duplicate_record_key_count":
        len(
            duplicate_record_keys
        ),

    "duplicate_record_keys":
        duplicate_record_keys,

    "missing_values_by_field":
        missing_values_by_field,

    "excluded_content_issue_count":
        len(
            excluded_content_issues
        ),

    "excluded_content_issues":
        excluded_content_issues,

    "structurally_evaluable":
        bool(structurally_evaluable)
}


TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_technical_diagnostics.json"
)


with open(
    TECHNICAL_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        TECHNICAL_DIAGNOSTICS,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 25. Experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        PDF_PATH.name,

    "source_sha256":
        PDF_SHA256,

    "source_verified":
        bool(
            ORIGINAL_PDF_INTEGRITY[
                "input_integrity_passed"
            ]
        ),

    "input_representation":
        "Original PDF document",

    "direct_document_ingestion":
        True,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "derived_representation_used_as_model_input":
        False,

    "extraction_scope_pages": [
        SOURCE_PAGE_START,
        SOURCE_PAGE_END
    ],

    "json_valid":
        bool(
            valid_json
        ),

    "structurally_evaluable":
        bool(structurally_evaluable),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        number_of_records,

    "record_count_matches":
        bool(
            record_count_valid
        ),

    "records_with_structure_issues":
        records_with_structure_issues,

    "records_with_type_issues":
        records_with_type_issues,

    "records_with_unexpected_units":
        len(
            unit_issues
        ),

    "records_with_unexpected_reference_periods":
        len(
            reference_period_issues
        ),

    "duplicate_record_key_count":
        len(
            duplicate_record_keys
        ),

    "excluded_content_issue_count":
        len(
            excluded_content_issues
        ),

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        bool(
            valid_json
        ),

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, "
        "Branch A direct-ingestion execution preservation, "
        "and technical output checks only. Agreement with "
        "the fixed Stage 1 reference dataset is evaluated "
        "in the separate Stage 4 validation."
    )
}


EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D3_branch_A_experiment_summary.json"
)


with open(
    EXPERIMENT_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_SUMMARY,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 26. Final experiment summary
# ============================================================

print(
    "=" * 48
)

print(
    "D3 Branch A experiment completed"
)

print(
    "=" * 48
)

print(
    f"Expected records        : "
    f"{EXPECTED_RECORD_COUNT}"
)

print(
    f"Observed records        : "
    f"{number_of_records}"
)

print(
    f"Valid JSON              : "
    f"{valid_json}"
)

print(
    f"Record count matches    : "
    f"{record_count_valid}"
)

print(
    f"Structurally evaluable  : "
    f"{structurally_evaluable}"
)

print(
    f"Structure issues        : "
    f"{records_with_structure_issues}"
)

print(
    f"Type issues             : "
    f"{records_with_type_issues}"
)

print(
    f"Unit issues             : "
    f"{len(unit_issues)}"
)

print(
    f"Reference-period issues : "
    f"{len(reference_period_issues)}"
)

print(
    f"Duplicate keys          : "
    f"{len(duplicate_record_keys)}"
)

print(
    f"Excluded-content issues : "
    f"{len(excluded_content_issues)}"
)

print()

print(
    "content_validation_performed: False"
)

print(
    "Next step: Validation A — D3"
)

In [ ]:
# ============================================================
# 27. Final artefact inventory
# ============================================================

GENERATED_OUTPUTS = [
    ORIGINAL_PDF_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if valid_json:

    GENERATED_OUTPUTS.insert(
        5,
        PARSED_EXTRACTION_PATH
    )


print(
    "Generated D3 Branch A files:\n"
)


for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name
    )

In [ ]:
# ============================================================
# 28. Download experiment artefacts
# ============================================================

for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            output_path
        )